# 3.Регрессионные модели
**Выполнено обучение моделей XGBoost, Random Forest,CatBoost,LightGBM для предсказания логарифма СC50. Для улучшения значения R² проведена настройка гиперпараметров с использованием метода случайного поиска (RandomizedSearchCV)**

In [ ]:
# установка библиотек
!pip install catboost lightgbm xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [ ]:
# подбор гиперпараметров XGBoost для SI

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_si.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_si.csv').values.ravel()

xgb = XGBRegressor(random_state=42, verbosity=0, early_stopping_rounds=10, eval_metric='rmse')

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

best_model = random_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("XGBoost SI Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {random_search.best_params_}")

joblib.dump(best_model, 'xgboost_si.pkl')

XGBoost SI Results:
R2: 0.2163
MAE_log: 0.9932
RMSE_log: 1.2658
MAE_original: 36.42
Best params: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


['xgboost_si.pkl']

In [ ]:
# подбор гиперпараметров Random Forest для SI

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_si.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_si.csv').values.ravel()

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_search.fit(X_train, y_train)

best_model = rf_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("Random Forest SI Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {rf_search.best_params_}")

joblib.dump(best_model, 'randomforest_si.pkl')

Random Forest SI Results:
R2: 0.2096
MAE_log: 0.9774
RMSE_log: 1.2712
MAE_original: 36.52
Best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20}


['randomforest_si.pkl']

In [ ]:
# подбор гиперпараметров CatBoost для SI

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_si.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_si.csv').values.ravel()

cat = CatBoostRegressor(random_seed=42, verbose=False, early_stopping_rounds=10)

param_dist_cat = {
    'iterations': [100, 200, 300],
    'depth': [3, 5, 7],
    'learning_rate': [0.03, 0.05, 0.1],
    'l2_leaf_reg': [3, 5, 7]
}

cat_search = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=15,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

cat_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

best_model = cat_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("CatBoost SI Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {cat_search.best_params_}")

joblib.dump(best_model, 'catboost_si.pkl')

CatBoost SI Results:
R2: 0.2569
MAE_log: 0.9577
RMSE_log: 1.2326
MAE_original: 34.70
Best params: {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 200, 'depth': 7}


['catboost_si.pkl']

In [ ]:
# подбор гиперпараметров LightGBM для SI

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_si.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_si.csv').values.ravel()

lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1)

param_dist_lgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'num_leaves': [31, 63, 127],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5]
}

lgb_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist_lgb,
    n_iter=15,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

lgb_search.fit(X_train, y_train)

best_model = lgb_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("LightGBM SI Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {lgb_search.best_params_}")

joblib.dump(best_model, 'lightgbm_si.pkl')

LightGBM SI Results:
R2: 0.2724
MAE_log: 0.9432
RMSE_log: 1.2196
MAE_original: 32.67
Best params: {'subsample': 0.9, 'reg_lambda': 0.1, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}


['lightgbm_si.pkl']

In [ ]:
# сравнение моделей для SI
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# загрузка тестовых данных
X_test = pd.read_csv('/content/X_test.csv')
y_test = pd.read_csv('/content/y_test_si.csv').values.ravel()

# загрузка моделей (только те, которые были обучены)
xgb_model = joblib.load('xgboost_si.pkl')
cat_model = joblib.load('catboost_si.pkl')
lgb_model = joblib.load('lightgbm_si.pkl')

print("="*60)
print("Сравнение моделей на тестовой выборке (SI)")
print("="*60)

# функция для расчёта метрик
def evaluate_model(model, name):
    y_pred_log = model.predict(X_test)

    r2 = r2_score(y_test, y_pred_log)
    mae_log = mean_absolute_error(y_test, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

    y_pred_orig = np.expm1(y_pred_log)
    y_test_orig = np.expm1(y_test)
    mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)

    print(f"\n{name}")
    print(f"  R2: {r2:.4f}")
    print(f"  MAE_log: {mae_log:.4f}")
    print(f"  RMSE_log: {rmse_log:.4f}")
    print(f"  MAE_original (нМ): {mae_orig:.2f}")

# оцениваем каждую модель
evaluate_model(xgb_model, "XGBoost")
evaluate_model(cat_model, "CatBoost")
evaluate_model(lgb_model, "LightGBM")

print("\n" + "="*60)
print("Лучшая модель по R2 и MAE_original:")

models_list = [
    ("XGBoost", xgb_model),
    ("CatBoost", cat_model),
    ("LightGBM", lgb_model)
]

best_r2 = -np.inf
best_mae = np.inf
best_name_r2 = ""
best_name_mae = ""

for name, model in models_list:
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    y_pred_orig = np.expm1(y_pred)
    y_test_orig = np.expm1(y_test)
    mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)

    if r2 > best_r2:
        best_r2 = r2
        best_name_r2 = name
    if mae_orig < best_mae:
        best_mae = mae_orig
        best_name_mae = name

print(f"  Лучшая по R2: {best_name_r2} ({best_r2:.4f})")
print(f"  Лучшая по MAE(нМ): {best_name_mae} ({best_mae:.2f})")

Сравнение моделей на тестовой выборке (SI)

XGBoost
  R2: 0.2163
  MAE_log: 0.9932
  RMSE_log: 1.2658
  MAE_original (нМ): 36.42

CatBoost
  R2: 0.2569
  MAE_log: 0.9577
  RMSE_log: 1.2326
  MAE_original (нМ): 34.70

LightGBM
  R2: 0.2724
  MAE_log: 0.9432
  RMSE_log: 1.2196
  MAE_original (нМ): 32.67

Лучшая модель по R2 и MAE_original:
  Лучшая по R2: LightGBM (0.2724)
  Лучшая по MAE(нМ): LightGBM (32.67)


Прогнозирование SI (индекс селективности = CC50 / IC50)
Лучшая модель: LightGBM
 (R2=0.2724,
MAE
ориг
​
 =32.67нМ).

Результат: Предсказание SI оказалось самой сложной задачей (
R
2
≈
0.27
R
2
 ≈0.27). Это ожидаемо, так как SI — это отношение двух величин, ошибки в предсказании которых накапливаются. Несмотря на низкий
R
2
, значение
MAE
ориг
​
 ≈33 может быть приемлемым, если пороговое значение селективности (например, SI > 8) находится далеко от этой ошибки.